# 🏥 3D DICOM Fine-Tuning for Multi-Label Chest Pathology Classification

**Objectif:** Fine-tune 3D-ResNet50 on 21 pathology classes using DICOM volumes with medical windowing

**Input:** DICOM files + dataset_labeled_enriched.csv

**Output:** Trained 3D model + metrics + per-class performance heatmaps

## 📦 Section 1: Install Dependencies and Import Libraries

In [ ]:
import subprocess
import sys

# Install required packages
packages = ['torch', 'torchvision', 'pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn', 'tqdm', 'pydicom', 'nibabel', 'scipy', 'monai']
for package in packages:
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models

import pandas as pd
import numpy as np
import os
from pathlib import Path
import pydicom
import nibabel as nib
from scipy import ndimage
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
import json
from datetime import datetime

print("✅ All packages imported successfully!")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 🔧 Section 2: Mount Google Drive and Configure Paths

In [ ]:
# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_FOLDER = '/content/drive/MyDrive/artishow'
    print("✅ Google Drive mounted")
except ImportError:
    # Local development
    DRIVE_FOLDER = '/home/infres/ahmed-25/artishow'
    print("⚠️ Running locally (no Google Drive)")

# Configure paths
VOLUMES_CSV = os.path.join(DRIVE_FOLDER, 'dataset_labeled_volumes_3d.csv')  # ✅ PREPROCESSED VOLUMES INDEX
LABELS_CSV = os.path.join(DRIVE_FOLDER, 'dataset_labeled_enriched.csv')
OUTPUT_FOLDER = os.path.join(DRIVE_FOLDER, 'dicom_volume', 'model_outputs_3d')

# Create output folder
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print(f"📁 VOLUMES INDEX (preprocessed): {VOLUMES_CSV}")
print(f"📊 Labels CSV (original): {LABELS_CSV}")
print(f"💾 Output folder: {OUTPUT_FOLDER}")

# Verify files exist
if os.path.exists(VOLUMES_CSV):
    print(f"✅ Preprocessed volumes CSV found!")
else:
    print(f"❌ Preprocessed volumes CSV NOT found at {VOLUMES_CSV}")
    print(f"   Run 'Preprocess_DICOM_to_Volumes_3D.ipynb' first!")

if os.path.exists(LABELS_CSV):
    print(f"✅ Labels CSV found!")
else:
    print(f"❌ Labels CSV NOT found at {LABELS_CSV}")

## 📊 Section 3: Load and Analyze DICOM Labels

In [ ]:
# Load preprocessed volumes index ✅
print("Loading preprocessed volumes index...")
df_labels = pd.read_csv(VOLUMES_CSV)

print(f"\n{'='*70}")
print("📊 PREPROCESSED VOLUMES DATASET OVERVIEW")
print(f"{'='*70}")
print(f"Total volumes available: {len(df_labels)}")
print(f"Columns: {list(df_labels.columns[:6])} ...")
print(f"\nFirst 3 rows:")
print(df_labels[['image_id', 'volume_npy_path', 'Cardiomegaly', 'Normal']].head(3))

# Verify volume paths exist
missing_volumes = []
for idx, row in df_labels.iterrows():
    volume_path = row['volume_npy_path']
    if not os.path.exists(volume_path):
        missing_volumes.append(volume_path)

if missing_volumes:
    print(f"\n⚠️ WARNING: {len(missing_volumes)} volume files missing!")
    # Remove rows with missing volumes
    df_labels = df_labels[df_labels['volume_npy_path'].apply(lambda x: os.path.exists(x))]
    print(f"   Kept {len(df_labels)} volumes with valid paths")
else:
    print(f"\n✅ All {len(df_labels)} volume files verified!")

# Define pathology classes
PATHOLOGY_CLASSES = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Pneumonia', 'Pneumothorax',
    'Edema', 'Emphysema', 'Fibrosis', 'Infiltration', 'Mass', 'Nodule',
    'Hernia', 'Fracture', 'Pleural_Thickening', 'Opacity', 'Consolidation',
    'Granuloma', 'Calcinosis', 'Scoliosis', 'Atherosclerosis', 'Normal'
]

# Analyze labels
print(f"\n{'='*70}")
print("🏷️  PATHOLOGY LABEL DISTRIBUTION (21 classes)")
print(f"{'='*70}")
print(f"Number of classes: {len(PATHOLOGY_CLASSES)}")
print(f"\nLabel prevalence:")

label_counts = df_labels[PATHOLOGY_CLASSES].sum().sort_values(ascending=False)
for label, count in label_counts.items():
    pct = count / len(df_labels) * 100
    print(f"  {label:20s}: {int(count):4d} ({pct:5.1f}%)")

## 📦 Section 4: DICOM Reading and 3D Volume Preprocessing

In [ ]:
def load_volume_npy(volume_path):
    """
    Load preprocessed volume from .npy file.
    Volumes are already windowed, resized, and normalized!
    """
    try:
        volume = np.load(volume_path)
        return volume
    except Exception as e:
        print(f"Error loading {volume_path}: {e}")
        return None

print("✅ Volume loading function defined")
print("\n📌 Note: Volumes are ALREADY preprocessed:")
print("   ✓ DICOM windowing applied (Hounsfield)")
print("   ✓ Resized to 128×128×64")
print("   ✓ Normalized (zero mean, unit variance)")

## 🔀 Section 5: Create Train/Val/Test Split

In [ ]:
# Dataset configuration
VOLUME_SIZE = (128, 128, 64)
BATCH_SIZE = 4  # Reduced for 3D volumes
NUM_WORKERS = 2

print(f"\n{'='*70}")
print("📊 Creating 3D DICOM Datasets (from preprocessed .npy files)")
print(f"{'='*70}")
print(f"Volume size: {VOLUME_SIZE}")

# Create datasets - ✅ NOW LOADING .NPY FILES
train_dataset = DICOM3DMultiLabelDataset(
    train_df, PATHOLOGY_CLASSES,
    transform=train_transforms
)
val_dataset = DICOM3DMultiLabelDataset(
    val_df, PATHOLOGY_CLASSES,
    transform=val_transforms
)
test_dataset = DICOM3DMultiLabelDataset(
    test_df, PATHOLOGY_CLASSES,
    transform=val_transforms
)

print(f"Train dataset: {len(train_dataset)}")
print(f"Val dataset: {len(val_dataset)}")
print(f"Test dataset: {len(test_dataset)}")

# Create dataloaders
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

print(f"\n✅ DataLoaders created (loading from .npy files)")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## 🏗️ Section 6: Define 3D PyTorch Dataset Class

## 🎨 Section 7: Define 3D Augmentations and Transforms

In [ ]:
class Random3DRotation(nn.Module):
    """Apply random 3D rotations to volumes"""
    def __init__(self, max_angle=15):
        super().__init__()
        self.max_angle = max_angle
    
    def forward(self, volume):
        if np.random.random() < 0.5:
            angle = np.random.uniform(-self.max_angle, self.max_angle)
            # Simple 2D rotation on each slice
            volume = torch.from_numpy(
                ndimage.rotate(volume.numpy(), angle, axes=(2, 3), reshape=False, order=1)
            ).float()
        return volume

class Random3DFlip(nn.Module):
    """Apply random 3D flips"""
    def __init__(self, p=0.5):
        super().__init__()
        self.p = p
    
    def forward(self, volume):
        # Random flip along height (dim 2)
        if np.random.random() < self.p:
            volume = torch.flip(volume, [2])
        # Random flip along width (dim 3)
        if np.random.random() < self.p:
            volume = torch.flip(volume, [3])
        return volume

# Training transforms (with augmentation)
train_transforms = nn.Sequential(
    Random3DFlip(p=0.3),
    Random3DRotation(max_angle=10)
)

# Validation/Test transforms (no augmentation)
val_transforms = nn.Identity()

print("✅ 3D transforms defined")
print(f"Train transforms: Flips + Rotations")
print(f"Val transforms: None (identity)")

## 📊 Section 8: Create DataLoaders

In [ ]:
# Dataset configuration
VOLUME_SIZE = (128, 128, 64)
BATCH_SIZE = 4  # Reduced for 3D volumes
NUM_WORKERS = 2

print(f"\n{'='*70}")
print("📊 Creating 3D DICOM Datasets")
print(f"{'='*70}")
print(f"Volume size: {VOLUME_SIZE}")

# Create datasets
train_dataset = DICOM3DMultiLabelDataset(
    train_df, DICOM_FOLDER, PATHOLOGY_CLASSES,
    volume_size=VOLUME_SIZE, transform=train_transforms, preload=False
)
val_dataset = DICOM3DMultiLabelDataset(
    val_df, DICOM_FOLDER, PATHOLOGY_CLASSES,
    volume_size=VOLUME_SIZE, transform=val_transforms, preload=False
)
test_dataset = DICOM3DMultiLabelDataset(
    test_df, DICOM_FOLDER, PATHOLOGY_CLASSES,
    volume_size=VOLUME_SIZE, transform=val_transforms, preload=False
)

print(f"Train dataset: {len(train_dataset)}")
print(f"Val dataset: {len(val_dataset)}")
print(f"Test dataset: {len(test_dataset)}")

# Create dataloaders
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

print(f"\n✅ DataLoaders created")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## 🤖 Section 9: Load and Configure 3D-ResNet50 Model

In [ ]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load 3D-ResNet50 from torchvision (pretrained on Kinetics)
print("\n📥 Loading 3D-ResNet50 (Kinetics pretrained)...")
try:
    model = models.video.r3d_18(pretrained=True)
    print("✅ Successfully loaded 3D-ResNet50")
except:
    print("⚠️ Using non-pretrained 3D-ResNet50")
    model = models.video.r3d_18(pretrained=False)

# Freeze early layers
for param in model.layer1.parameters():
    param.requires_grad = False
for param in model.layer2.parameters():
    param.requires_grad = False

# Replace classifier for MULTI-LABEL (21 outputs)
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(num_ftrs, len(PATHOLOGY_CLASSES))  # 21 outputs
)

model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✅ Model configured for MULTI-LABEL classification")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Output shape: [batch_size, {len(PATHOLOGY_CLASSES)}]")

## 🏋️ Section 10: Training Setup

In [ ]:
EPOCHS = 5
LEARNING_RATE = 1e-4
PATIENCE = 3

# ✅ MULTI-LABEL: Use BCEWithLogitsLoss
criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=1e-5
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, verbose=True
)

history = {
    'train_loss': [],
    'val_loss': [],
}

best_val_loss = float('inf')
patience_counter = 0

print(f"{'='*70}")
print("🏋️  TRAINING CONFIGURATION (3D MULTI-LABEL)")
print(f"{'='*70}")
print(f"Epochs: {EPOCHS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Loss function: BCEWithLogitsLoss ✅")
print(f"Model: 3D-ResNet50")
print(f"Device: {device}")
print(f"Early stopping patience: {PATIENCE}")

## 🔄 Section 11: Training Loop

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device, epoch, num_epochs):
    """Train one epoch for 3D multi-label classification"""
    model.train()
    total_loss = 0
    
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{num_epochs} [TRAIN]")
    for volumes, labels, _ in pbar:
        volumes = volumes.to(device)  # [B, 1, D, H, W]
        labels = labels.to(device)    # [B, 21]
        
        # Forward pass
        outputs = model(volumes)  # [B, 21]
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': loss.item():.4f})
    
    return total_loss / len(loader)

def validate(model, loader, criterion, device):
    """Validate one epoch"""
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        pbar = tqdm(loader, desc="[VAL]")
        for volumes, labels, _ in pbar:
            volumes = volumes.to(device)
            labels = labels.to(device)
            
            outputs = model(volumes)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            pbar.set_postfix({'loss': loss.item():.4f})
    
    return total_loss / len(loader)

print("✅ Training functions defined")

## 🏗️ Section 6: Define 3D PyTorch Dataset Class

In [ ]:
class DICOM3DMultiLabelDataset(Dataset):
    """Multi-label 3D Dataset loading preprocessed .npy volumes"""
    
    def __init__(self, dataframe, label_columns, transform=None):
        """
        Args:
            dataframe: DataFrame with 'volume_npy_path' + label columns
            label_columns: List of 21 pathology column names
            transform: Augmentation pipeline
        """
        self.dataframe = dataframe.reset_index(drop=True)
        self.label_columns = label_columns
        self.transform = transform
        self.volumes_cache = {}
    
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        volume_path = row['volume_npy_path']
        image_id = row['image_id']
        
        # Load preprocessed volume from .npy
        volume = load_volume_npy(volume_path)
        
        if volume is None:
            # Return zero volume if loading fails
            volume = np.zeros((128, 128, 64), dtype=np.float32)
        
        # Convert to tensor and add channel dimension
        volume_tensor = torch.from_numpy(volume).float().unsqueeze(0)  # [1, H, W, D]
        
        # Apply transforms (augmentation)
        if self.transform:
            volume_tensor = self.transform(volume_tensor)
        
        # Get labels (21 binary values)
        labels = torch.tensor(row[self.label_columns].values.astype(np.float32))
        
        return volume_tensor, labels, image_id

print("✅ 3D Dataset class defined (loads .npy volumes)")
print(f"   Input: Preprocessed .npy files (already normalized)")
print(f"   Output: [1, 128, 128, 64] volume tensor + [21] labels")

In [ ]:
# Main training loop
print(f"\n{'='*70}")
print("🚀 STARTING 3D MODEL TRAINING")
print(f"{'='*70}\n")

for epoch in range(EPOCHS):
    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device, epoch, EPOCHS)
    history['train_loss'].append(train_loss)
    
    # Validate
    val_loss = validate(model, val_loader, criterion, device)
    history['val_loss'].append(val_loss)
    
    # LR scheduler
    scheduler.step(val_loss)
    
    # Print metrics
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        model_path = os.path.join(OUTPUT_FOLDER, 'best_model_3d.pth')
        torch.save(model.state_dict(), model_path)
        print(f"✅ Best model saved! (Loss: {val_loss:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n⏹️  Early stopping triggered")
            break

print(f"\n{'='*70}")
print("✅ TRAINING COMPLETE!")
print(f"{'='*70}")

## 📊 Section 12: Training History Visualization

In [ ]:
# Plot training history
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(history['train_loss'], label='Train Loss', marker='o', linewidth=2)
ax.plot(history['val_loss'], label='Val Loss', marker='s', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('3D Model Training History', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER, 'training_history_3d.png'), dpi=100, bbox_inches='tight')
plt.show()

print(f"✅ Training history plot saved!")

## 🧪 Section 13: Test Set Evaluation

In [ ]:
# Load best model
model_path = os.path.join(OUTPUT_FOLDER, 'best_model_3d.pth')
model.load_state_dict(torch.load(model_path))

# Test evaluation
model.eval()
all_preds = []
all_probs = []
all_labels = []

print("\n🧪 Evaluating on test set...")
with torch.no_grad():
    for volumes, labels, _ in tqdm(test_loader):
        volumes = volumes.to(device)
        outputs = model(volumes)
        
        # Get probabilities and predictions
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).int()
        
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.numpy())

# Convert to arrays
all_preds = np.array(all_preds)
all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

# Calculate metrics
test_acc = accuracy_score(all_labels, all_preds)
test_precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
test_recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
test_f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

print(f"\n{'='*70}")
print("📊 TEST SET RESULTS")
print(f"{'='*70}")
print(f"Accuracy:  {test_acc*100:.2f}%")
print(f"Precision: {test_precision*100:.2f}%")
print(f"Recall:    {test_recall*100:.2f}%")
print(f"F1-Score:  {test_f1*100:.2f}%")

## 🎯 Section 14: Per-Pathology Performance Metrics

In [ ]:
# Calculate per-pathology metrics
per_pathology_metrics = {}

for i, pathology in enumerate(PATHOLOGY_CLASSES):
    true_labels_i = all_labels[:, i]
    predicted_labels_i = all_preds[:, i]
    probs_i = all_probs[:, i]
    
    acc = accuracy_score(true_labels_i, predicted_labels_i)
    prec = precision_score(true_labels_i, predicted_labels_i, pos_label=1, zero_division=0)
    rec = recall_score(true_labels_i, predicted_labels_i, pos_label=1, zero_division=0)
    f1 = f1_score(true_labels_i, predicted_labels_i, pos_label=1, zero_division=0)
    
    # AUC if positive labels exist
    if len(np.unique(true_labels_i)) > 1:
        auc = roc_auc_score(true_labels_i, probs_i)
    else:
        auc = 0.0
    
    per_pathology_metrics[pathology] = {
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'auc': auc
    }

# Create heatmap
metrics_data = []
for pathology in PATHOLOGY_CLASSES:
    metrics = per_pathology_metrics[pathology]
    metrics_data.append([
        metrics['accuracy'],
        metrics['precision'],
        metrics['recall'],
        metrics['f1'],
        metrics['auc']
    ])

metrics_array = np.array(metrics_data)

# Plot heatmap
fig, ax = plt.subplots(figsize=(10, 14))
im = ax.imshow(metrics_array, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)

ax.set_xticks(np.arange(5))
ax.set_yticks(np.arange(len(PATHOLOGY_CLASSES)))
ax.set_xticklabels(['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC'])
ax.set_yticklabels(PATHOLOGY_CLASSES)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Score', rotation=270, labelpad=15)

# Add values to heatmap
for i in range(len(PATHOLOGY_CLASSES)):
    for j in range(5):
        text = ax.text(j, i, f'{metrics_array[i, j]:.2f}',
                       ha="center", va="center", color="black", fontsize=8)

ax.set_title('Per-Pathology Performance Metrics (3D Model)', fontsize=14, pad=20)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER, 'per_pathology_metrics_3d.png'), dpi=100, bbox_inches='tight')
plt.show()

print(f"✅ Per-pathology metrics heatmap saved!")

## 💾 Section 15: Save Model and Generate Report

## 📈 Section 16: Final Summary Report

In [ ]:
summary = f"""
{'='*80}
✨ 3D DICOM FINE-TUNING SUMMARY - 3D-ResNet50 (Kinetics Pretrained)
{'='*80}

📊 DATASET INFORMATION:
   Total DICOM studies: {len(df_labels)}
   Training set: {len(train_df)} ({len(train_df)/len(df_labels)*100:.1f}%)
   Validation set: {len(val_df)} ({len(val_df)/len(df_labels)*100:.1f}%)
   Test set: {len(test_df)} ({len(test_df)/len(df_labels)*100:.1f}%)
   Number of pathology classes: {len(PATHOLOGY_CLASSES)}
   Volume dimensions: {VOLUME_SIZE[0]}×{VOLUME_SIZE[1]}×{VOLUME_SIZE[2]}

🤖 MODEL INFORMATION:
   Architecture: 3D-ResNet50
   Pretrained on: Kinetics-400
   Total parameters: {total_params:,}
   Trainable parameters: {trainable_params:,}
   Output: {len(PATHOLOGY_CLASSES)} binary pathology predictions

⚙️  TRAINING CONFIGURATION:
   Epochs trained: {len(history['train_loss'])}
   Batch size: {BATCH_SIZE}
   Learning rate: {LEARNING_RATE}
   Optimizer: Adam (weight_decay=1e-5)
   Loss function: BCEWithLogitsLoss (multi-label)
   Scheduler: ReduceLROnPlateau
   Early stopping patience: {PATIENCE}

📈 RESULTS:
   Test Accuracy: {test_acc*100:.2f}%
   Test Precision: {test_precision*100:.2f}%
   Test Recall: {test_recall*100:.2f}%
   Test F1-Score: {test_f1*100:.2f}%

💾 OUTPUT FILES:
   - model_3d_final.pth (final model weights)
   - best_model_3d.pth (best validation checkpoint)
   - config_3d.json (model configuration)
   - per_pathology_metrics_3d.json (per-class metrics)
   - training_history_3d.png (loss curves)
   - per_pathology_metrics_3d.png (performance heatmap)
   - split_train_3d.csv, split_val_3d.csv, split_test_3d.csv (data splits)

🎯 NEXT STEPS:
   1. Load model and make predictions on new DICOM studies
   2. Analyze per-pathology performance for clinical validation
   3. Fine-tune hyperparameters (learning rate, epochs, batch size)
   4. Experiment with 3D data augmentations
   5. Combine with radiologist reports for multi-modal learning

🏆 TOP PERFORMING PATHOLOGIES (by F1-Score):
"""

# Add top pathologies
sorted_pathologies = sorted(
    per_pathology_metrics.items(),
    key=lambda x: x[1]['f1'],
    reverse=True
)[:5]

for i, (pathology, metrics) in enumerate(sorted_pathologies, 1):
    summary += f"   {i}. {pathology:20s} (F1={metrics['f1']:.3f}, AUC={metrics['auc']:.3f})\n"

summary += f"\n{'='*80}\n"

print(summary)

# Save summary
with open(os.path.join(OUTPUT_FOLDER, 'training_summary_3d.txt'), 'w') as f:
    f.write(summary)

print(f"✅ Summary saved to {os.path.join(OUTPUT_FOLDER, 'training_summary_3d.txt')}")